In [42]:
import numpy as np
import pandas as pd
import os
from src.data_loader import prepare_paths, load_horizontal_well, load_typewell
from src.features import (generate_signal_features, generate_spatial_features,
        integrate_with_typewells, full_preprocessing_pipeline)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from src.prepare_data import prepare_target
from src.prepare_data import train_test_split_paths
from sklearn.metrics import root_mean_squared_error
from src.model_utils import validate_model

In [43]:
model_xgb = xgb.XGBRegressor()
model_xgb.load_model('xgb_model.json')

In [44]:
paths_test_str = 'test'
paths_test = prepare_paths(paths_test_str)

In [45]:
test = []

for path in paths_test:
    new_df = full_preprocessing_pipeline(*path)
    test.append(new_df)

/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lag_{lag}'] = df['GR'].shift(lag).fillna(method='bfill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lead_{lag}'] = df['GR'].shift(-lag).fillna(method='ffill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[f'gr_lag_{lag}'] = df['GR'].shift(lag).fillna(method='bfill')
/Users/iaroslav/Downloads/wellbore-geology-prediction/src/features.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.f

In [46]:
test_prepared = []

for frame in test:
    prepared_frame = frame.copy()
    prepared_frame["TVT_prev"] = prepared_frame["TVT_input"].shift(1).fillna(0)
    test_prepared.append(prepared_frame)



In [47]:
test_concated = pd.concat(test_prepared, ignore_index=True)
test_concated.head()

,MD,X,Y,Z,GR,TVT_input,gr_roll_mean_5,gr_roll_std_5,gr_roll_min_5,gr_roll_max_5,...,gr_lag_3,gr_lead_3,delta_X,delta_Y,delta_Z,trajectory_slope,tw_reference_GR,tw_reference_TVT,gr_difference_to_tw,TVT_prev
0,11467.0,2983525.16,1069022.09,-9258.57,115.692586,11236.02,122.241280,11.436583,115.584293,135.446960,...,115.692586,140.401346,0.00,0.00,0.00,0.000000,115.53,11340.45,0.162586,0.00
1,11468.0,2983525.18,1069022.30,-9259.55,115.584293,11237.05,126.781296,13.024744,115.584293,140.401346,...,115.692586,111.270638,0.02,0.21,-0.98,-4.645624,115.53,11340.45,0.054293,11236.02
2,11469.0,2983525.20,1069022.52,-9260.52,135.446960,11238.09,123.679165,13.241943,111.270638,140.401346,...,115.692586,108.779909,0.02,0.22,-0.97,-4.390964,135.14,11275.45,0.306960,11237.05
3,11470.0,2983525.22,1069022.73,-9261.50,140.401346,11239.12,122.296629,14.577737,108.779909,140.401346,...,115.692586,114.338929,0.02,0.21,-0.98,-4.645624,139.44,11252.95,0.961346,11238.09
4,11471.0,2983525.25,1069022.95,-9262.47,111.270638,11240.15,122.047556,14.730928,108.779909,140.401346,...,115.584293,121.459167,0.03,0.22,-0.97,-4.368641,111.25,11318.45,0.020638,11239.12


In [48]:
tvt_input = test_concated["TVT_input"]
test_no_input = test_concated.drop(columns=["TVT_input"])

predictions = model_xgb.predict(test_no_input)

In [49]:
pred_input = pd.DataFrame({
    "TVT_input": tvt_input,
    "pred": predictions,
})
pred_input["pred"] = pred_input["TVT_input"].where(pred_input["TVT_input"].notna(), pred_input["pred"])
pred_input.head(2)

,TVT_input,pred
0,11236.02,11236.02
1,11237.05,11237.05


In [50]:
true_predictions = pred_input["pred"]
true_predictions.head(2)

0    11236.02
1    11237.05
Name: pred, dtype: float64

In [51]:
path_df = {}

for i in range(len(paths_test)):
    horizontal_path, _ = paths_test[i]
    path_df[horizontal_path] = test[i]

total_rows = 0

output_rows = []
for horizontal_path, value in path_df.items():

    well_id = os.path.basename(horizontal_path).replace("__horizontal_well.csv", "")
    for row_idx in range(len(value)):
        output_rows.append({
            "id": f"{well_id}_{row_idx}",
            "tvt": float(true_predictions[total_rows])
        })
        total_rows += 1


output = pd.DataFrame(output_rows)
output.tail()

,id,tvt
19216,00e12e8b_6379,10256.112305
19217,00e12e8b_6380,10256.112305
19218,00e12e8b_6381,10256.178711
19219,00e12e8b_6382,10256.178711
19220,00e12e8b_6383,10256.178711


In [52]:
output.to_csv("submission.csv", index=False)